# Tests GH 25-06 data reader and time series checker

This notebook reads the CSV files from `Tests GH 25-06`, parses the LoRa export format, synchronizes manual group windows with the CSV timestamps when needed, checks mapping quality, and plots RSSI/SNR through time for visual inspection.


In [1]:
# If you run this in Google Colab, authentication is handled in the next cell.
# If you run locally with downloaded CSVs, set LOCAL_DATA_DIR in the config cell.

from __future__ import annotations

import io
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# ---- Configuration ----

DRIVE_FOLDER_ID = "1Egq_nPkzHTInTvNQbOo5O5w5wvAbIAec"  # Tests GH 25-06
LOCAL_DATA_DIR = "D:/Users/user/Documents/GatewayPlacement/TestsGH2506"
ONLY_CSV_FILES = True
CACHE_DIR = Path("data_cache/tests_gh_25_06")
MANUAL_TIME_RANGES_PATH = Path("manual_time_ranges.json")

# Manual windows use [start_time, next_start_time - buffer].
# The final group starts at the last registered manual time and remains open-ended.
END_BEFORE_NEXT_START_SECONDS = 8

# Structures whose CSV clock is already aligned with the manual JSON times.
ABSOLUTE_CLOCK_STRUCTURES = {"R1354", "R1356", "R1185", "R1187", "R1189", "R1352"}

# Structures whose manual schedule is shifted so the first manual start equals the first CSV timestamp.
CSV_ANCHOR_STRUCTURES = {"R1358", "R1183", "R1360", "R1181"}

# QC only: this does not assign positions. Change exceptions if the protocol notes change.
EXPECTED_GROUPS_DEFAULT = 12
EXPECTED_GROUPS_EXCEPTIONS = {"R1354": 11}


In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def build_drive_service():
    """Create an authenticated Google Drive v3 service in Colab or a local env."""
    try:
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
    except Exception as exc:
        raise RuntimeError(
            "Missing Google API packages. In Colab they are usually available. "
            "Locally, install google-api-python-client google-auth google-auth-oauthlib."
        ) from exc

    if running_in_colab():
        from google.colab import auth  # type: ignore
        auth.authenticate_user()
        import google.auth
        creds, _ = google.auth.default()
        return build("drive", "v3", credentials=creds), MediaIoBaseDownload

    # Local fallback: use Application Default Credentials if configured with gcloud auth application-default login.
    import google.auth
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    return build("drive", "v3", credentials=creds), MediaIoBaseDownload


def list_drive_csv_files(folder_id: str) -> pd.DataFrame:
    service, _ = build_drive_service()
    rows = []
    page_token = None
    query = f"'{folder_id}' in parents and trashed = false"
    if ONLY_CSV_FILES:
        query += " and mimeType = 'text/csv'"
    while True:
        resp = service.files().list(
            q=query,
            fields="nextPageToken, files(id, name, mimeType, size, createdTime, modifiedTime)",
            orderBy="modifiedTime",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        rows.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return pd.DataFrame(rows)


def download_drive_file_text(file_id: str) -> str:
    service, MediaIoBaseDownload = build_drive_service()
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return fh.getvalue().decode("utf-8-sig", errors="replace")


def load_file_catalog() -> pd.DataFrame:
    if LOCAL_DATA_DIR:
        paths = sorted(Path(LOCAL_DATA_DIR).glob("*.csv"))
        return pd.DataFrame({
            "id": [None] * len(paths),
            "name": [p.name for p in paths],
            "path": [str(p) for p in paths],
            "mimeType": ["text/csv"] * len(paths),
            "modifiedTime": [pd.Timestamp(p.stat().st_mtime, unit="s").isoformat() for p in paths],
        })
    return list_drive_csv_files(DRIVE_FOLDER_ID)


def read_file_text(row: pd.Series) -> str:
    if LOCAL_DATA_DIR:
        return Path(row["path"]).read_text(encoding="utf-8-sig", errors="replace")

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = CACHE_DIR / f"{row['id']}_{row['name']}"
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8-sig", errors="replace")
    text = download_drive_file_text(row["id"])
    cache_path.write_text(text, encoding="utf-8")
    return text

In [ ]:
def parse_filename_metadata(name: str) -> dict:
    stem = Path(name).stem
    ids = [int(x) for x in re.findall(r"(?<!\d)(\d{3,5})(?!\d)", stem)]
    primary_id = ids[0] if ids else None
    mirror_match = re.search(r"mirror\s*([0-9]{3,5})", stem, flags=re.I)
    mirror_id = int(mirror_match.group(1)) if mirror_match else None
    date_match = re.search(r"(\d{1,2})[-_](\d{1,2})", stem)
    test_date_hint = None
    if date_match:
        test_date_hint = f"2026-{int(date_match.group(2)):02d}-{int(date_match.group(1)):02d}"
    return {
        "file_stem": stem,
        "primary_id": primary_id,
        "mirror_id": mirror_id,
        "test_date_hint": test_date_hint,
    }


def parse_details(details: object) -> dict:
    if pd.isna(details):
        return {}
    parsed = {}
    for line in str(details).splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip().lower().replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
        parsed[f"detail_{key}"] = value.strip()
    return parsed


def parse_detail_json(value: object):
    if pd.isna(value) or not str(value).strip():
        return None
    raw = str(value).strip()
    try:
        return json.loads(raw)
    except Exception:
        try:
            return json.loads(raw.replace('""', '"'))
        except Exception:
            return raw


def split_rssi_snr(series: pd.Series) -> pd.DataFrame:
    split = series.fillna("-/").astype(str).str.strip().str.split("/", n=1, expand=True)
    if split.shape[1] == 1:
        split[1] = np.nan
    return pd.DataFrame({
        "rssi": pd.to_numeric(split[0].replace({"-": np.nan, "": np.nan}), errors="coerce"),
        "snr": pd.to_numeric(split[1].replace({"-": np.nan, "": np.nan}), errors="coerce"),
    })


def parse_signal_csv(text: str, file_row: pd.Series) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(io.StringIO(text), engine="python")
    df.columns = [c.strip() for c in df.columns]
    name = file_row.get("name", file_row.get("title", "unknown.csv"))
    meta = parse_filename_metadata(name)
    df["source_file"] = name
    df["source_id"] = file_row.get("id", None)
    for key, value in meta.items():
        df[key] = value

    df = df.rename(columns={
        "Device EUI/Group": "device_eui_group",
        "Gateway ID": "gateway_id",
        "RSSI/SNR": "rssi_snr",
        "Fcnt": "fcnt",
        "Type": "type",
        "Time": "time",
        "Details": "details",
        "Frequency": "frequency",
        "Datarate": "datarate",
        "Size": "size",
    })

    df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)
    df["time_local"] = df["time"].dt.tz_convert("Europe/Amsterdam")
    df["fcnt"] = pd.to_numeric(df.get("fcnt"), errors="coerce")
    df["frequency"] = pd.to_numeric(df.get("frequency"), errors="coerce")
    df["size"] = pd.to_numeric(df.get("size"), errors="coerce")
    df = pd.concat([df, split_rssi_snr(df.get("rssi_snr", pd.Series(index=df.index, dtype=object)))], axis=1)

    details_df = pd.DataFrame([parse_details(x) for x in df.get("details", pd.Series(index=df.index, dtype=object))])
    if not details_df.empty:
        df = pd.concat([df.reset_index(drop=True), details_df.reset_index(drop=True)], axis=1)
    df["port"] = pd.to_numeric(df.get("detail_port"), errors="coerce")
    df["payload_hex"] = df.get("detail_payloadhex", pd.Series(index=df.index, dtype=object))
    df["detail_json_obj"] = df.get("detail_json", pd.Series(index=df.index, dtype=object)).apply(parse_detail_json)
    df["gpio_in_2"] = df["detail_json_obj"].apply(lambda x: x.get("gpio_in_2") if isinstance(x, dict) else np.nan)
    df["message_text"] = df["detail_json_obj"].apply(lambda x: x.get("text") if isinstance(x, dict) else None)

    parse_info = {
        "measurable_uplink_rows": int((df["time"].notna() & df["rssi"].notna() & df["snr"].notna() & df["type"].astype(str).str.contains("Up", case=False, na=False)).sum())
    }
    return df.sort_values("time"), parse_info

In [ ]:
catalog = load_file_catalog()
catalog = catalog.sort_values("modifiedTime", na_position="last").reset_index(drop=True)
catalog

In [ ]:
frames = []
segment_reports = []

for _, row in catalog.iterrows():
    text = read_file_text(row)
    df_one, seg_info = parse_signal_csv(text, row)
    frames.append(df_one)
    segment_reports.append({
        "source_file": row.get("name", row.get("title", "unknown.csv")),
        "rows": len(df_one),
        "uplink_rows": int(df_one["type"].astype(str).str.contains("Up", case=False, na=False).sum()),
        "measurable_rows": int(df_one[["rssi", "snr"]].notna().all(axis=1).sum()),
        "first_time": df_one["time_local"].min(),
        "last_time": df_one["time_local"].max(),
        **seg_info,
    })

data = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
segment_report = pd.DataFrame(segment_reports)
segment_report

In [ ]:
file_summary = (
    data.groupby(["source_file", "primary_id", "mirror_id"], dropna=False)
    .agg(
        rows=("source_file", "size"),
        first_time=("time_local", "min"),
        last_time=("time_local", "max"),
        uplink_rows=("type", lambda s: s.astype(str).str.contains("Up", case=False, na=False).sum()),
        rssi_median=("rssi", "median"),
        snr_median=("snr", "median"),
    )
    .reset_index()
)
file_summary


In [ ]:
def plot_file_timeseries(df: pd.DataFrame, file_name: str, group_col: str = "manual_group"):
    one = df[df["source_file"] == file_name].sort_values("time_local").copy()
    measured = one[one["type"].astype(str).str.contains("Up", case=False, na=False)]
    if measured.empty:
        print(f"No uplink measurements for {file_name}")
        return

    fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True, gridspec_kw={"height_ratios": [2, 2, 1]})
    fig.suptitle(file_name, y=0.98, fontsize=14)

    group_values = measured[group_col].dropna().unique() if group_col in measured.columns else []
    for group in group_values:
        chunk = measured[measured[group_col] == group]
        if chunk.empty:
            continue
        axes[0].axvspan(chunk["time_local"].min(), chunk["time_local"].max(), alpha=0.055, color="tab:blue")
        axes[1].axvspan(chunk["time_local"].min(), chunk["time_local"].max(), alpha=0.055, color="tab:blue")

    for port, chunk in measured.groupby("port", dropna=False):
        label = f"Port {int(port)}" if pd.notna(port) else "Port ?"
        axes[0].scatter(chunk["time_local"], chunk["rssi"], s=24, label=label, alpha=0.82)
        axes[1].scatter(chunk["time_local"], chunk["snr"], s=24, label=label, alpha=0.82)

    gpio = measured[measured["gpio_in_2"].notna()]
    if not gpio.empty:
        axes[2].step(gpio["time_local"], gpio["gpio_in_2"], where="post", color="tab:green", linewidth=1.8, label="gpio_in_2")
        axes[2].scatter(gpio["time_local"], gpio["gpio_in_2"], s=18, color="tab:green")
    else:
        axes[2].text(0.01, 0.5, "No gpio_in_2 JSON payloads detected", transform=axes[2].transAxes, va="center")

    axes[0].set_ylabel("RSSI (dBm)")
    axes[1].set_ylabel("SNR (dB)")
    axes[2].set_ylabel("GPIO")
    axes[2].set_xlabel("Time (Europe/Amsterdam)")
    axes[0].legend(loc="best", ncol=4)
    axes[2].set_yticks([0, 1])
    plt.tight_layout()
    plt.show()


# Raw per-file plotting helper. Run manually after manual assignment if needed:
# plot_file_timeseries(data, catalog["name"].iloc[0])

## Manual Time Mapping

Manual times are used absolutely for the structures whose clocks agree. For the shifted structures, the whole manual schedule is moved by `csv_first_time - first_manual_start_time`. After the last registered manual time, the notebook creates one final open-ended group.


In [ ]:
MANUAL_CONFIG = json.loads(MANUAL_TIME_RANGES_PATH.read_text(encoding="utf-8"))
TEST_DATE = "2026-06-25"
LOCAL_TZ = "Europe/Amsterdam"


def hhmmss_to_timestamp(value, date=TEST_DATE, tz=LOCAL_TZ):
    if value is None or pd.isna(value):
        return pd.NaT
    return pd.Timestamp(f"{date} {value}", tz=tz)


def manual_ranges_to_frame(config: dict) -> pd.DataFrame:
    rows = []
    for key, (start, next_start) in config["ranges"].items():
        structure, group = key.split("_G", 1)
        manual_start = hhmmss_to_timestamp(start)
        manual_next_start = hhmmss_to_timestamp(next_start)
        rows.append({
            "range_key": key,
            "structure_id": structure,
            "manual_group": int(group),
            "start_raw": start,
            "next_start_raw": next_start,
            "manual_start_time": manual_start,
            "manual_next_start_time": manual_next_start,
        })
    ranges = pd.DataFrame(rows).sort_values(["structure_id", "manual_group"]).reset_index(drop=True)
    ranges["manual_duration_to_next_s"] = (ranges["manual_next_start_time"] - ranges["manual_start_time"]).dt.total_seconds()
    return ranges


def structure_from_row(row: pd.Series) -> str | None:
    value = row.get("primary_id")
    if pd.isna(value):
        value = row.get("mirror_id")
    if pd.isna(value):
        return None
    return f"R{int(value)}"


def sync_policy_for(structure_id: str) -> str:
    if structure_id in CSV_ANCHOR_STRUCTURES:
        return "csv_anchor"
    return "absolute_time"


raw_data = data.copy()
raw_data["structure_id"] = raw_data.apply(structure_from_row, axis=1)
manual_ranges = manual_ranges_to_frame(MANUAL_CONFIG)

csv_first_times = (
    raw_data.dropna(subset=["time_local", "structure_id"])
    .groupby("structure_id", as_index=False)
    .agg(csv_first_time=("time_local", "min"))
)
manual_first_times = (
    manual_ranges.dropna(subset=["manual_start_time"])
    .groupby("structure_id", as_index=False)
    .agg(manual_first_time=("manual_start_time", "min"))
)
sync_offsets = manual_first_times.merge(csv_first_times, on="structure_id", how="outer")
sync_offsets["sync_policy"] = sync_offsets["structure_id"].map(sync_policy_for)
sync_offsets["csv_anchor_offset"] = sync_offsets["csv_first_time"] - sync_offsets["manual_first_time"]
sync_offsets["applied_offset"] = pd.Timedelta(0)
csv_anchor_mask = sync_offsets["sync_policy"].eq("csv_anchor")
sync_offsets.loc[csv_anchor_mask, "applied_offset"] = sync_offsets.loc[csv_anchor_mask, "csv_anchor_offset"]
sync_offsets["applied_offset"] = pd.to_timedelta(sync_offsets["applied_offset"])
sync_offsets["applied_offset_s"] = sync_offsets["applied_offset"].dt.total_seconds()
sync_offsets


In [ ]:
def build_mapping_windows(ranges: pd.DataFrame, offsets: pd.DataFrame) -> pd.DataFrame:
    windows = ranges.merge(
        offsets[["structure_id", "csv_first_time", "manual_first_time", "sync_policy", "applied_offset"]],
        on="structure_id",
        how="left",
    )
    windows["applied_offset"] = pd.to_timedelta(windows["applied_offset"].fillna(pd.Timedelta(0)))
    windows["start_time"] = windows.apply(lambda row: row["manual_start_time"] + row["applied_offset"] if pd.notna(row["manual_start_time"]) else pd.NaT, axis=1)
    windows["next_start_time"] = windows.apply(lambda row: row["manual_next_start_time"] + row["applied_offset"] if pd.notna(row["manual_next_start_time"]) else pd.NaT, axis=1)
    windows["valid_end"] = windows["next_start_time"].apply(lambda value: value - pd.Timedelta(seconds=END_BEFORE_NEXT_START_SECONDS) if pd.notna(value) else pd.NaT)
    windows["is_final_group"] = False
    windows["has_complete_window"] = (
        windows["start_time"].notna()
        & windows["valid_end"].notna()
        & (windows["valid_end"] > windows["start_time"])
    )

    final_rows = []
    for structure_id, group in windows.groupby("structure_id", dropna=False):
        registered_times = pd.concat([group["manual_start_time"], group["manual_next_start_time"]]).dropna()
        if registered_times.empty:
            continue
        final_manual_start = registered_times.max()
        final_group = int(group["manual_group"].max()) + 1
        offset = group["applied_offset"].dropna().iloc[0] if group["applied_offset"].notna().any() else pd.Timedelta(0)
        final_rows.append({
            "range_key": f"{structure_id}_G{final_group}_final",
            "structure_id": structure_id,
            "manual_group": final_group,
            "start_raw": None,
            "next_start_raw": None,
            "manual_start_time": final_manual_start,
            "manual_next_start_time": pd.NaT,
            "manual_duration_to_next_s": np.nan,
            "csv_first_time": group["csv_first_time"].dropna().iloc[0] if group["csv_first_time"].notna().any() else pd.NaT,
            "manual_first_time": group["manual_first_time"].dropna().iloc[0] if group["manual_first_time"].notna().any() else pd.NaT,
            "sync_policy": group["sync_policy"].dropna().iloc[0] if group["sync_policy"].notna().any() else sync_policy_for(structure_id),
            "applied_offset": offset,
            "start_time": final_manual_start + offset,
            "next_start_time": pd.NaT,
            "valid_end": pd.NaT,
            "is_final_group": True,
            "has_complete_window": pd.notna(final_manual_start),
        })

    if final_rows:
        windows = pd.concat([windows, pd.DataFrame(final_rows)], ignore_index=True)

    windows["window_duration_s"] = windows.apply(
        lambda row: (row["valid_end"] - row["start_time"]).total_seconds() if pd.notna(row["valid_end"]) and pd.notna(row["start_time"]) else np.nan,
        axis=1,
    )
    windows.loc[windows["is_final_group"], "window_duration_s"] = np.nan
    windows.loc[~windows["has_complete_window"], "window_duration_s"] = np.nan
    return windows.sort_values(["structure_id", "manual_group", "is_final_group"]).reset_index(drop=True)


mapping_windows = build_mapping_windows(manual_ranges, sync_offsets)
mapping_windows[[
    "sync_policy", "structure_id", "manual_group", "manual_start_time", "csv_first_time",
    "start_time", "valid_end", "is_final_group", "has_complete_window"
]].head(30)


In [ ]:
def match_manual_group(time_value, structure_id: str | None, windows: pd.DataFrame):
    if structure_id is None or pd.isna(structure_id):
        return pd.Series({"manual_group": pd.NA, "manual_group_status": "file_structure_unknown", "manual_group_reliable": False})
    if pd.isna(time_value):
        return pd.Series({"manual_group": pd.NA, "manual_group_status": "missing_time", "manual_group_reliable": False})

    sub = windows[(windows["structure_id"] == structure_id) & (windows["has_complete_window"])]
    if sub.empty:
        return pd.Series({"manual_group": pd.NA, "manual_group_status": "no_complete_manual_window_for_structure", "manual_group_reliable": False})

    regular = sub[~sub["is_final_group"]]
    matches = regular[(regular["start_time"] <= time_value) & (time_value < regular["valid_end"])]

    if matches.empty:
        final = sub[sub["is_final_group"]]
        matches = final[final["start_time"] <= time_value]

    if len(matches) == 1:
        status = "assigned_final_group" if bool(matches.iloc[0]["is_final_group"]) else "assigned_manual_window"
        return pd.Series({"manual_group": int(matches.iloc[0]["manual_group"]), "manual_group_status": status, "manual_group_reliable": True})
    if len(matches) > 1:
        groups = ",".join(matches["manual_group"].astype(str))
        return pd.Series({"manual_group": pd.NA, "manual_group_status": f"ambiguous_manual_window_overlap:{groups}", "manual_group_reliable": False})

    nearest = sub.iloc[(sub["start_time"] - time_value).abs().argsort()[:1]]
    if not nearest.empty:
        delta_s = abs((time_value - nearest.iloc[0]["start_time"]).total_seconds())
        return pd.Series({"manual_group": pd.NA, "manual_group_status": f"outside_manual_window_nearest_G{int(nearest.iloc[0]['manual_group'])}_{delta_s:.1f}s", "manual_group_reliable": False})
    return pd.Series({"manual_group": pd.NA, "manual_group_status": "outside_manual_windows", "manual_group_reliable": False})


def add_manual_groups(df: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    assigned = out.apply(lambda row: match_manual_group(row["time_local"], row["structure_id"], windows), axis=1)
    out = pd.concat([out, assigned], axis=1)
    out["manual_group"] = out["manual_group"].astype("Int64")
    out["manual_group_reliable"] = out["manual_group_reliable"].astype(bool)
    out = out.merge(sync_offsets[["structure_id", "sync_policy", "applied_offset_s"]], on="structure_id", how="left")
    return out


mapped_data = add_manual_groups(raw_data, mapping_windows)
data = mapped_data

manual_assignment_report = (
    mapped_data.groupby(["source_file", "structure_id", "sync_policy", "manual_group_status", "manual_group_reliable"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source_file", "manual_group_reliable", "manual_group_status"], ascending=[True, False, True])
)
manual_assignment_report


In [ ]:
def expected_groups_for(structure_id: str) -> int:
    return EXPECTED_GROUPS_EXCEPTIONS.get(structure_id, EXPECTED_GROUPS_DEFAULT)


def summarize_manual_groups(df: pd.DataFrame) -> pd.DataFrame:
    measured = df[df["type"].astype(str).str.contains("Up", case=False, na=False)].copy()
    measured = measured[measured["manual_group"].notna()]
    if measured.empty:
        return pd.DataFrame()

    def mad(series):
        series = series.dropna()
        if series.empty:
            return np.nan
        return float((series - series.median()).abs().median())

    return (
        measured.groupby(["structure_id", "sync_policy", "manual_group", "manual_group_status"], dropna=False)
        .agg(
            samples=("rssi", "count"),
            first_time=("time_local", "min"),
            last_time=("time_local", "max"),
            rssi_median=("rssi", "median"),
            rssi_mad=("rssi", mad),
            snr_median=("snr", "median"),
            snr_mad=("snr", mad),
        )
        .reset_index()
        .sort_values(["structure_id", "manual_group"])
    )


manual_group_summary = summarize_manual_groups(mapped_data)

expected_group_report = (
    manual_group_summary.groupby(["structure_id", "sync_policy"], dropna=False)
    .agg(assigned_groups=("manual_group", "nunique"), samples=("samples", "sum"))
    .reset_index()
)
expected_group_report["expected_groups"] = expected_group_report["structure_id"].map(expected_groups_for)
expected_group_report["missing_groups"] = expected_group_report["expected_groups"] - expected_group_report["assigned_groups"]
expected_group_report["mapping_count_ok"] = expected_group_report["missing_groups"].eq(0)

manual_group_summary, expected_group_report


In [ ]:
def plot_manual_timeseries(df: pd.DataFrame, structure_id: str, metric_cols=("rssi", "snr")):
    one = df[(df["structure_id"] == structure_id) & df["type"].astype(str).str.contains("Up", case=False, na=False)].copy()
    one = one.sort_values("time_local")
    if one.empty:
        print(f"No uplink rows found for {structure_id}")
        return
    policy = one["sync_policy"].dropna().iloc[0] if one["sync_policy"].notna().any() else "unknown"
    fig, axes = plt.subplots(len(metric_cols) + 1, 1, figsize=(15, 3.2 * (len(metric_cols) + 1)), sharex=True)
    fig.suptitle(f"{structure_id}: synchronized manual windows ({policy})", fontsize=14, y=0.995)
    ranges = mapping_windows[mapping_windows["structure_id"] == structure_id]
    for ax in axes[:-1]:
        for _, rr in ranges.iterrows():
            if not bool(rr["has_complete_window"]):
                continue
            if bool(rr["is_final_group"]):
                ax.axvline(rr["start_time"], color="tab:red", alpha=0.45, linestyle="--")
            else:
                ax.axvspan(rr["start_time"], rr["valid_end"], color="tab:blue", alpha=0.08)
            ax.text(rr["start_time"], 0.98, f"G{int(rr['manual_group'])}", transform=ax.get_xaxis_transform(), va="top", fontsize=8, alpha=0.75)
    reliable = one[one["manual_group_reliable"]]
    unreliable = one[~one["manual_group_reliable"]]
    for ax, metric in zip(axes[:-1], metric_cols):
        for group, chunk in reliable.groupby("manual_group"):
            ax.scatter(chunk["time_local"], chunk[metric], s=24, label=f"G{int(group)}", alpha=0.86)
        if not unreliable.empty:
            ax.scatter(unreliable["time_local"], unreliable[metric], s=30, facecolors="none", edgecolors="black", label="outside manual window")
        ax.set_ylabel(metric.upper())
        ax.legend(loc="best", ncol=6, fontsize=8)
    gpio = one[one["gpio_in_2"].notna()]
    if not gpio.empty:
        axes[-1].step(gpio["time_local"], gpio["gpio_in_2"], where="post", color="tab:green")
        axes[-1].scatter(gpio["time_local"], gpio["gpio_in_2"], s=18, color="tab:green")
    axes[-1].set_ylabel("GPIO")
    axes[-1].set_xlabel("Time (Europe/Amsterdam)")
    axes[-1].set_yticks([0, 1])
    plt.tight_layout()
    plt.show()


for structure_id in sorted(mapped_data["structure_id"].dropna().unique()):
    plot_manual_timeseries(mapped_data, structure_id)
